In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pypdf import PdfReader

c:\Users\sreek\EnterPriseKnowLedgeAssitant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
Pages = []

reader = PdfReader("C:/Users/sreek/EnterPriseKnowLedgeAssitant/data/Attntion.pdf")

for page in reader.pages:
    Pages.append(page.extract_text())

print(len(Pages))

15


In [3]:
#splitting the text into chunks

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
    )




In [4]:
Chunks = []

for i, page in enumerate(Pages):
    chunks = text_splitter.split_text(page)
    for j, chunk in enumerate(chunks):
        Chunks.append(chunk)

print(len(Chunks))    

161


In [5]:
from huggingface_hub import InferenceClient
import os
from dotenv import load_dotenv

load_dotenv()

client = InferenceClient(
    provider="hf-inference",
    api_key=os.getenv("hf_token")
)

embedding = client.feature_extraction(
    "What is multi-head attention?",
    model="BAAI/bge-small-en-v1.5"
)

print(type(embedding))
print(len(embedding))

<class 'numpy.ndarray'>
384


In [6]:
def generate_embedding(text: str):
    return client.feature_extraction(
        text,
        model="BAAI/bge-small-en-v1.5"
    )

In [7]:
import psycopg
from dotenv import load_dotenv
import os
from pinecone import Pinecone, ServerlessSpec
load_dotenv()

pinecone_api_key = os.getenv("pinecone_api_key")
pc = Pinecone(api_key=pinecone_api_key)


pc.delete_index("attntion-384")

index_name = "attntion-384"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
 
pc = pc.Index(index_name)


In [8]:
vectors = []

for i, chunk in enumerate(Chunks):
    embedding = generate_embedding(chunk)
    metadata = {"chunk_index": i, "text": chunk}
    vectors.append((f"chunk_{i}", embedding, metadata))

print("All chunks stored!", len(vectors))

All chunks stored! 161


In [9]:
print("Records stored in Pinecone index:", vectors[0])

Records stored in Pinecone index: ('chunk_0', array([-1.15970534e-03, -1.28781749e-02, -3.00010890e-02, -5.80141656e-02,
        5.98012023e-02,  7.40287006e-02, -9.55944136e-03,  5.42873442e-02,
        1.25417311e-03,  3.59666497e-02, -2.22218931e-02, -1.42188184e-02,
        4.72557135e-02,  1.51825938e-02, -2.63872254e-03,  4.80413511e-02,
       -5.43215163e-02, -2.26055849e-02,  5.47100184e-03,  2.19881441e-02,
        5.09429500e-02, -6.46465048e-02,  5.80349825e-02,  1.52714159e-02,
        1.77154206e-02,  1.73717570e-02, -1.44956242e-02, -8.42932016e-02,
       -2.03336701e-02, -1.62522793e-01, -4.81005106e-03, -4.90775257e-02,
       -1.38961179e-02, -4.75559710e-03, -1.80329140e-02, -2.46801749e-02,
        1.27627347e-02,  5.45315724e-03, -1.41182747e-02,  1.63723398e-02,
       -6.00188551e-03,  1.32405786e-02, -4.08175401e-02, -4.74965246e-03,
        3.63874286e-02, -1.78633071e-02, -4.54913527e-02, -1.01636630e-02,
       -5.84501366e-04, -3.17702852e-02, -4.18311208e-

In [10]:
print("Upserting vectors into Pinecone index...")
pc.upsert(vectors=vectors)

print("Vectors upserted successfully!")

Upserting vectors into Pinecone index...
Vectors upserted successfully!
